# SCRAPPING ARTKEL DAN KONTEN

Jalankan semua instalasi

In [1]:
!pip install feedparser newspaper3k pandas

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 55.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 7.5 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=63d24a242a9b3a0b0a2fbaaec0dc66f6da9778c1f42e8425b540c1d47db77c5f
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=809af46656d8ceb283b94c7a8a61792815733c7f18aec8ac87cbf32f3b36a25d
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f16

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install feedparser pandas requests beautifulsoup4

In [4]:
!pip install newspaper3k lxml_html_clean

import newspaper
from newspaper import news_pool
from newspaper import Article

Scrapping Link artikel

In [ ]:
import feedparser
import pandas as pd
import time

queries = [
    "TikTok Shop Indonesia",
    "TikTok Shop UMKM",
    "TikTok Shop regulasi",
    "TikTok Shop Tokopedia",
    "Tiktok Shop E-commerce"
    "TikTok Shop",
    "Dampak TikTok Shop",
    "UMKM Bangkrut",
    "TikTok Shop UMKM gulung tikar",
    "TikTok Shop bunuh UMKM",
    "Tiktok UMKM",
    "Dampak TikTokShop",
    "Tiktok Shop larangan",
    "Efek Tiktok Shop",
    "Manfaat Tiktok Shop",
    "Keuntungan Tiktok Shop",
    "Tiktok Shop merugikan",
    "TikTok Shop matikan pasar tradisional"

]


all_articles = []

for query in queries:
    print("Mengambil:", query)

    rss_url = f"https://news.google.com/rss/search?q={query.replace(' ', '+')}&hl=id&gl=ID&ceid=ID:id"
    feed = feedparser.parse(rss_url)

    for entry in feed.entries:

        direct_link = ""
        if "links" in entry:
            for link in entry.links:
                if link.type == "text/html":
                    direct_link = link.href
                    break

        all_articles.append({
            "title": entry.title,
            "source": entry.source.title if "source" in entry else "",
            "link": direct_link,
            "published": pd.to_datetime(entry.published) if "published" in entry else None,
            "category": query
        })

    time.sleep(1)

df = pd.DataFrame(all_articles)

# Hapus duplikat berdasarkan title
df = df.drop_duplicates(subset=["title"])

print("Total setelah hapus duplikat:", len(df))

file_name = "tiktok_shop_articles.xlsx"
df.to_excel(file_name, index=False)

print("Final jumlah:", len(df))
print("File tersimpan:", file_name)
display(df.head())

In [ ]:
df.to_excel("tiktok_shop_news.xlsx", index=False)

file ditambahkan link direct ke artikel/berita

beautifulsoup4 scraping content

**upload file artikel_raw.csv ke drive terlebih dahulu**

In [1]:
!pip install beautifulsoup4 requests pandas
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

file_path = '/content/drive/MyDrive/artikel_raw.csv'

try:
    df_input = pd.read_csv(file_path)
    print(f"File '{file_path}' berhasil dimuat. Total: {len(df_input)} baris.")
except FileNotFoundError:
    print(f"Error: File tidak ditemukan di {file_path}. Pastikan file sudah diupload ke folder utama My Drive.")
    df_input = pd.DataFrame()

def scrape_content(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7',
        'Referer': 'https://www.google.com/'
    }

    filter_keywords = ["ADVERTISEMENT", "SCROLL TO CONTINUE", "BACA JUGA", "SIMAK JUGA", "VIDEO TERKAIT"]

    try:
        time.sleep(random.uniform(0.5, 1.5))
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            for noise in soup(['script', 'style', 'header', 'footer', 'nav', 'aside', 'iframe']):
                noise.decompose()

            paragraphs = soup.find_all('p')
            clean_paragraphs = [p.text.strip() for p in paragraphs if p.text.strip() and not any(k.upper() in p.text.upper() for k in filter_keywords)]

            content = "\n".join(clean_paragraphs)
            return content, True if content.strip() else False
        return None, False
    except:
        return None, False

if not df_input.empty:
    processed_data = []
    success_count = 0
    fail_count = 0

    print(f"Memulai scraping seluruh data ({len(df_input)} baris)...\n")

    for index, row in df_input.iterrows():
        url = row['link asli']
        new_row = row.to_dict()
        new_row['content'] = None
        new_row['scrape_status'] = 'Failed'

        if pd.isna(url) or not str(url).startswith('http'):
            fail_count += 1
            new_row['scrape_status'] = 'Skipped (Invalid URL)'
            processed_data.append(new_row)
            continue

        if index % 10 == 0 or index == len(df_input) - 1:
            print(f"Progress: {index+1}/{len(df_input)}... Berhasil: {success_count}")

        content, is_success = scrape_content(url)

        new_row['content'] = content
        if is_success:
            new_row['scrape_status'] = 'Success'
            success_count += 1
        else:
            new_row['scrape_status'] = 'Failed'
            fail_count += 1

        processed_data.append(new_row)

    df_final = pd.DataFrame(processed_data)

    if not df_final.empty:
        output_file = 'scraped_articles_final2.csv'
        df_final.to_csv(output_file, index=False)
        print(f"\nPROSES SELESAI!")
        print(f"Jumlah artikel berhasil discrape: {success_count}")
        print(f"Jumlah artikel gagal/diskim: {fail_count}")
        print(f"Total artikel diproses: {len(df_final)}")
        print(f"File disimpan: {output_file}")
        display(df_final.head())
    else:
        print("\nTidak ada data yang berhasil diambil atau diproses.")

File '/content/drive/MyDrive/artikel_raw.csv' berhasil dimuat. Total: 1053 baris.
Memulai scraping seluruh data (1053 baris)...

Progress: 1/1053... Berhasil: 0
Progress: 11/1053... Berhasil: 7
Progress: 21/1053... Berhasil: 11
Progress: 31/1053... Berhasil: 17
Progress: 41/1053... Berhasil: 22
Progress: 51/1053... Berhasil: 31
Progress: 61/1053... Berhasil: 40
Progress: 71/1053... Berhasil: 45
Progress: 81/1053... Berhasil: 51
Progress: 91/1053... Berhasil: 58
Progress: 101/1053... Berhasil: 64
Progress: 111/1053... Berhasil: 72
Progress: 121/1053... Berhasil: 81
Progress: 131/1053... Berhasil: 88
Progress: 141/1053... Berhasil: 95
Progress: 151/1053... Berhasil: 103
Progress: 161/1053... Berhasil: 111
Progress: 171/1053... Berhasil: 120
Progress: 181/1053... Berhasil: 127
Progress: 191/1053... Berhasil: 135
Progress: 201/1053... Berhasil: 142
Progress: 211/1053... Berhasil: 149
Progress: 221/1053... Berhasil: 157
Progress: 231/1053... Berhasil: 166
Progress: 241/1053... Berhasil: 175

,title,source,link asli,published,content,category,sentimen,scrape_status
0,TikTok raises minimum user age in Indonesia fo...,ANTARA News Bali,https://bali.antaranews.com/berita/403417/tikt...,2026-04-14 11:19:15,Jakarta (ANTARA) -\nSocial media giant TikTok ...,Regulasi,Netral,Success
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,Jawa Pos,https://www.jawapos.com/aplikasi/2310040011/ti...,2026-04-12 16:57:43,Fitur TikTok Shop resmi ditutup hari ini./\nJa...,Isu,Negatif,Success
2,Konten sebagai Penggerak Kinerja: Strategi Pen...,Neraca.co.id,https://www.neraca.co.id/article/235666/konten...,2026-04-11 15:50:09,None,Strategi,Positif,Failed
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,SWA.co.id,https://swa.co.id/read/471326/live-tokopediati...,2026-04-12 14:45:00,Kementerian Perdagangan mencatat nilai penjual...,Bisnis,Positif,Success
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,kontan.co.id,https://industri.kontan.co.id/news/konten-jadi...,2026-04-11 14:02:56,Reporter: Dina Mirayanti Hutauruk | Editor: Di...,Strategi,Positif,Success


In [ ]:
df_final.to_excel("scraped_final.xlsx", index=False)

scrapping versi newspaper untuk melengkapi yang failed

In [ ]:
import nltk
nltk.download('punkt_tab')
import newspaper
from newspaper import Config
import pandas as pd
import os

file_path = 'artikel_raw.csv'

if not os.path.exists(file_path):
    print(f"Error: File tidak ditemukan di {file_path}. Pastikan file sudah diupload.")
else:
    df_full = pd.read_csv(file_path)
    df_orig = df_full.copy()

    config = Config()
    config.request_timeout = 20
    config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'

    all_contents = []
    successful_count = 0
    failed_count = 0

    print(f"Memproses {len(df_orig)} artikel...")

    for index, row in df_orig.iterrows():
        url = row['link asli']
        text = None

        if pd.notna(url) and str(url).startswith('http'):
            try:
                article = newspaper.Article(url=url, language='id', config=config)
                article.download()
                article.parse()

                text = article.text

                print(f"[{index+1}/{len(df_orig)}] Berhasil: {url[:50]}...")
                successful_count += 1

            except Exception as e:
                print(f"[{index+1}/{len(df_orig)}] Gagal memproses {url[:50]}... Error: {str(e)}")
                failed_count += 1
        else:
            print(f"[{index+1}/{len(df_orig)}] Skip: Tautan tidak valid atau kosong: {url}")
            failed_count += 1

        all_contents.append(text)

    df_orig['content'] = all_contents

    output_file = 'scraped articles_ver3.csv'
    df_orig.to_csv(output_file, index=False)

    print(f"\nProses selesai! File disimpan sebagai: {output_file}")
    print(f"Total berhasil diambil: {successful_count}")
    print(f"Total gagal diambil: {failed_count}")
    display(df_orig.head())

merging kedua csv (mengisi kekosongan di versi beautifulsoup)

In [3]:
import pandas as pd

file_final2 = 'scraped_articles_final2.csv'
file_ver3 = 'scraped articles_ver3.csv'

try:
    df2 = pd.read_csv(file_final2)
    df3 = pd.read_csv(file_ver3)

    df2_indexed = df2.set_index('link asli')
    df3_indexed = df3.set_index('link asli')

    df2_indexed['content'] = df2_indexed['content'].combine_first(df3_indexed['content'])

    df_result = df2_indexed.reset_index()

    df_result = df_result[df2.columns]

    output_file = 'scraped_articles_merged.csv'
    df_result.to_csv(output_file, index=False)

    print(f"Berhasil menggabungkan data")
    print(f"File tersimpan sebagai: {output_file}")

    null_count = df_result['content'].isna().sum()
    print(f"Jumlah baris dengan konten kosong sekarang: {null_count}")
    display(df_result.head())

except Exception as e:
    print(f"Terjadi kesalahan: {e}")

Berhasil menggabungkan data!
File tersimpan sebagai: scraped_articles_merged.csv
Jumlah baris dengan konten kosong sekarang: 249


,title,source,link asli,published,content,category,sentimen,scrape_status
0,TikTok raises minimum user age in Indonesia fo...,ANTARA News Bali,https://bali.antaranews.com/berita/403417/tikt...,2026-04-14 11:19:15,Jakarta (ANTARA) -\nSocial media giant TikTok ...,Regulasi,Netral,Success
1,TikTok Indonesia Resmi Umumkan Tutup Fitur Tik...,Jawa Pos,https://www.jawapos.com/aplikasi/2310040011/ti...,2026-04-12 16:57:43,Fitur TikTok Shop resmi ditutup hari ini./\nJa...,Isu,Negatif,Success
2,Konten sebagai Penggerak Kinerja: Strategi Pen...,Neraca.co.id,https://www.neraca.co.id/article/235666/konten...,2026-04-11 15:50:09,NaN,Strategi,Positif,Failed
3,LIVE Tokopedia–TikTok Shop Ditonton 38 Miliar ...,SWA.co.id,https://swa.co.id/read/471326/live-tokopediati...,2026-04-12 14:45:00,Kementerian Perdagangan mencatat nilai penjual...,Bisnis,Positif,Success
4,Konten Jadi Mesin Penggerak Penjualan di Tokop...,kontan.co.id,https://industri.kontan.co.id/news/konten-jadi...,2026-04-11 14:02:56,Reporter: Dina Mirayanti Hutauruk | Editor: Di...,Strategi,Positif,Success
